In [1]:
#Day19_Persistent_ID_Tracking_and_Continuous_Video_Loop

In [2]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.9 MB/s eta 0:00:00


In [3]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image as PILImage
from ultralytics import YOLO
import pandas as pd
model = YOLO('yolov8n.pt')
NaN = np.nan

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
cap = cv2.VideoCapture("/content/football_clip.mp4")
ret,frame = cap.read()
ret2, frame2 = cap.read()   # grab the next frame after the one already used

In [5]:
def get_centroids(frame):
  result = model.predict(frame)
  boxes = result[0].boxes.xyxy.cpu().numpy()
  centroid = []

  for box in boxes :
    x1,y1,x2,y2 = box
    cx = (x1+x2)/2
    cy = (y1+y2)/2
    centroid.append((cx,cy))
  return centroid

centroid_frame1 = get_centroids(frame)
centroid_frame2 = get_centroids(frame2)




0: 384x640 7 persons, 321.5ms
Speed: 14.7ms preprocess, 321.5ms inference, 39.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 persons, 140.5ms
Speed: 5.7ms preprocess, 140.5ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)


In [6]:
print(centroid_frame1)
print(centroid_frame2)

[(np.float32(1309.5), np.float32(1181.5613)), (np.float32(2138.3008), np.float32(1163.9364)), (np.float32(3153.3193), np.float32(1180.0847)), (np.float32(2373.5881), np.float32(1165.3231)), (np.float32(1808.2808), np.float32(1148.4467)), (np.float32(1200.4038), np.float32(1154.464)), (np.float32(3742.0144), np.float32(1186.4226))]
[(np.float32(1310.1368), np.float32(1180.262)), (np.float32(2138.9731), np.float32(1163.5256)), (np.float32(3153.0117), np.float32(1179.8584)), (np.float32(2374.056), np.float32(1165.1177)), (np.float32(1808.4924), np.float32(1148.9598)), (np.float32(1198.5328), np.float32(1153.611)), (np.float32(3742.6655), np.float32(1186.3446))]


In [7]:
import math
point = centroid_frame2[0]
matches = []

for point in centroid_frame2:
  best_distance = float('inf')   # nothing can be closer than "infinitely far", so first real d always wins
  best_match = None

  for candidate in centroid_frame1:
    d = math.dist(point,candidate)
    if d<best_distance:
      best_distance = d       # replace with the new smaller distance
      best_match = candidate  # remember which point gave us that distance
  matches.append((point,best_match))

print(matches)


[((np.float32(1310.1368), np.float32(1180.262)), (np.float32(1309.5), np.float32(1181.5613))), ((np.float32(2138.9731), np.float32(1163.5256)), (np.float32(2138.3008), np.float32(1163.9364))), ((np.float32(3153.0117), np.float32(1179.8584)), (np.float32(3153.3193), np.float32(1180.0847))), ((np.float32(2374.056), np.float32(1165.1177)), (np.float32(2373.5881), np.float32(1165.3231))), ((np.float32(1808.4924), np.float32(1148.9598)), (np.float32(1808.2808), np.float32(1148.4467))), ((np.float32(1198.5328), np.float32(1153.611)), (np.float32(1200.4038), np.float32(1154.464))), ((np.float32(3742.6655), np.float32(1186.3446)), (np.float32(3742.0144), np.float32(1186.4226)))]


In [8]:
print(matches)

[((np.float32(1310.1368), np.float32(1180.262)), (np.float32(1309.5), np.float32(1181.5613))), ((np.float32(2138.9731), np.float32(1163.5256)), (np.float32(2138.3008), np.float32(1163.9364))), ((np.float32(3153.0117), np.float32(1179.8584)), (np.float32(3153.3193), np.float32(1180.0847))), ((np.float32(2374.056), np.float32(1165.1177)), (np.float32(2373.5881), np.float32(1165.3231))), ((np.float32(1808.4924), np.float32(1148.9598)), (np.float32(1808.2808), np.float32(1148.4467))), ((np.float32(1198.5328), np.float32(1153.611)), (np.float32(1200.4038), np.float32(1154.464))), ((np.float32(3742.6655), np.float32(1186.3446)), (np.float32(3742.0144), np.float32(1186.4226)))]


In [9]:
# **Day 18: Cross-Frame Player Matching — Nearest Centroid Tracking**

# - Detected all players in a frame using YOLO, filtered for the "person" class
# - Reduced each player's bounding box to a single centroid point
# - Wrapped detection + centroid logic into a reusable `get_centroids(frame)` function
# - Grabbed a second frame and computed its centroids too
# - For each player in frame 2, found the closest-distance centroid in frame 1 using `math.dist`
# - Produced a matched list linking each frame-2 player to their frame-1 position
# - This nearest-neighbor matching is the foundation for real tracking —
#    ID assignment, speed, and possession all build on it next

In [10]:
players_position = {}
next_id = 0

for c in centroid_frame1:
  players_position[next_id] = c
  next_id +=1

print(players_position)

{0: (np.float32(1309.5), np.float32(1181.5613)), 1: (np.float32(2138.3008), np.float32(1163.9364)), 2: (np.float32(3153.3193), np.float32(1180.0847)), 3: (np.float32(2373.5881), np.float32(1165.3231)), 4: (np.float32(1808.2808), np.float32(1148.4467)), 5: (np.float32(1200.4038), np.float32(1154.464)), 6: (np.float32(3742.0144), np.float32(1186.4226))}


In [11]:
max_distance = 50
for c in centroid_frame2:
  best_id = None
  best_distance = float("inf")

  for pid,pos in players_position.items():
    d = math.dist(c,pos)
    if d < best_distance :
      best_distance = d
      best_id = pid
  if best_distance < max_distance:
    players_position[best_id] = c
  else:
    players_position[best_id] = c
    next_id += 1
print(players_position)

{0: (np.float32(1310.1368), np.float32(1180.262)), 1: (np.float32(2138.9731), np.float32(1163.5256)), 2: (np.float32(3153.0117), np.float32(1179.8584)), 3: (np.float32(2374.056), np.float32(1165.1177)), 4: (np.float32(1808.4924), np.float32(1148.9598)), 5: (np.float32(1198.5328), np.float32(1153.611)), 6: (np.float32(3742.6655), np.float32(1186.3446))}


In [11]:
# **Day 19: Persistent ID Tracking and Continuous Video Loop**

# **Conclusion:**
# - Built a `player_positions` dict ({id: last_centroid}) that persists across frames instead of resetting each time
# - Added nearest-neighbor matching against this dict, updating an existing ID when
#      a close-enough match is found
# - Added a `max_distance` threshold — matches beyond it are treated as a new
#     player and get a fresh ID, correctly handling players entering or leaving the frame
# - Planned extending this into a continuous `while` loop with `cap.read()`,
#      running until the video ends (`ret == False`), so IDs persist across
#       the whole clip instead of just two test frames
# - Result: a working foundation for real player tracking — everything downstream
#     (per-player speed, distance covered, possession) builds on these stable IDs